# SPRSound training workflow

This notebook demonstrates the training workflow for the AST model the SPRSound publicly available dataset.

It converts SPRSound-specific files into the generic metadata CSVs consumed by the AST preparation script, then runs training and evaluation.

> Note: this notebook is not meant to demo model generalization capabilities on SPRSound dataset as the dataset is rather small for a binary classification task that we are using here. The purpose is to provide an example of an end-to-end training and inference pipeline that can be reproduced and used to work with other datasets.

Expected layout: this repository is `transformers-paper`, and the SPRSound dataset directory is next to it as `../SPRSound`.

In [ ]:
from pathlib import Path
from datetime import datetime
import os
from tqdm import tqdm

from IPython.display import display
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

scripts_dir = project_root / "scripts"

# Dataset location
sprsound_root = project_root.parent / "SPRSound"

# Training run folder configs
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
work_dir = project_root / "temp" / "sprsound_training" / run_id
metadata_dir = work_dir / "metadata"
ast_dir = work_dir / "prepared_ast"
model_dir = work_dir / "models"
predictions_dir = work_dir / "predictions"

# Training run hyperparameters configs
ast_hparams_path = project_root / "hparams" / "ast_asthma.yaml"

metadata_dir.mkdir(parents=True, exist_ok=False)
model_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)

assert sprsound_root.exists(), f"SPRSound not found: {sprsound_root}"
assert ast_hparams_path.exists(), f"AST hparams not found: {ast_hparams_path}"

project_root, sprsound_root, work_dir

(PosixPath('/workspace/transformers-paper'),
 PosixPath('/workspace/SPRSound'),
 PosixPath('/workspace/transformers-paper/temp/sprsound_training/20260603_165557'))

SPRSound labels are converted to a binary screening task:

- control patients become `healthy`
- patients with a known disease become `pathological`

Rows without a usable disease label are excluded. The train/eval split is made using the 5-second clips as the ISU.

In [4]:
# Helper function
def target_label_for_disease(disease):
    """Map one SPRSound disease label to the binary training label."""
    normalized = str(disease).strip().casefold()
    if normalized in {"", "-", "nan"}:
        return None
    return "healthy" if normalized == "control group" else "pathological"

def cap_majority_ratio(
    rows,
    majority_label="pathological",
    minority_label="healthy",
    ratio=2,
    random_state=42,
):
    """Downsample the majority label to a fixed majority:minority row ratio.

    Note: this is implemented in the notebook because SPRSound is imbalanced
          for the label split we use in this demo.
    """
    majority = rows[rows["target_label"] == majority_label]
    minority = rows[rows["target_label"] == minority_label]
    max_majority = min(len(majority), ratio * len(minority))
    majority = majority.sample(n=max_majority, random_state=random_state)
    return (
        pd.concat([majority, minority], ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .sort_values("sample_id")
    )

In [5]:
summary_path = sprsound_root / "Patient Summary" / "SPRSound_patient_summary.csv"
audio_dir = sprsound_root / "Classification" / "train_classification_wav"

site_map = {
    "p1": "left posterior",
    "p2": "left lateral",
    "p3": "right posterior",
    "p4": "right lateral",
    "p5": "left anterior",
    "p6": "right anterior",
    "p7": "left lower lateral",
    "p8": "right lower lateral",
}

summary = pd.read_csv(summary_path)
summary["patient_id"] = summary["patient_num"].astype(int).astype(str)
summary["target_label"] = summary["disease"].map(target_label_for_disease)
summary = summary.dropna(subset=["target_label"])

label_by_patient = dict(zip(summary["patient_id"], summary["target_label"]))

rows = []
for wav_path in tqdm(sorted(audio_dir.glob("*.wav"))):
    patient_id, age_years, sex_key, site_key, _recording_id = wav_path.stem.split("_")
    if patient_id not in label_by_patient:
        continue
    rows.append(
        {
            "patient_id": patient_id,
            "sample_id": wav_path.stem,
            "audio_path": os.path.relpath(wav_path, metadata_dir),
            "target_label": label_by_patient[patient_id],
            "sex": "M" if sex_key == "0" else "F",
            "age_years": float(age_years),
            "recording_site": site_map[site_key],
        }
    )

metadata = pd.DataFrame(rows)

source_metadata = metadata.drop(columns=["patient_id"]).sort_values("sample_id")
metadata_csv = metadata_dir / "metadata.csv"
source_metadata.to_csv(metadata_csv, index=False)

display(source_metadata.head())
source_metadata["target_label"].value_counts()

100%|██████████| 1949/1949 [00:00<00:00, 31191.62it/s]


,sample_id,audio_path,target_label,sex,age_years,recording_site
0,40069321_15.3_0_p1_981,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,left posterior
1,40069321_15.3_0_p2_982,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,left lateral
2,40069321_15.3_0_p3_983,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,right posterior
3,40069321_15.3_0_p4_984,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,right lateral
4,40138127_14.7_0_p1_137,../../../../../SPRSound/Classification/train_c...,pathological,M,14.7,left posterior


target_label
pathological    1816
healthy          133
Name: count, dtype: int64

The exported CSVs use the generic metadata contract. AST needs `sample_id`, `audio_path`, and `target_label`.

In [6]:
pd.read_csv(metadata_csv).head()

,sample_id,audio_path,target_label,sex,age_years,recording_site
0,40069321_15.3_0_p1_981,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,left posterior
1,40069321_15.3_0_p2_982,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,left lateral
2,40069321_15.3_0_p3_983,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,right posterior
3,40069321_15.3_0_p4_984,../../../../../SPRSound/Classification/train_c...,pathological,M,15.3,right lateral
4,40138127_14.7_0_p1_137,../../../../../SPRSound/Classification/train_c...,pathological,M,14.7,left posterior


Prepare AST feature tensors from the generic metadata CSVs.

In [7]:
!python {scripts_dir}/prepare_ast_dataset.py \
    --metadata_csv {metadata_csv} \
    --output_dir {ast_dir} \
    --dataset_name sprsound_clips

100%|███████████████████████████████████████| 1949/1949 [04:36<00:00,  7.04it/s]


In [8]:
ast_clips_dir = ast_dir / "sprsound_clips"
ast_features_dir = ast_clips_dir / "features"
ast_metadata_csv = ast_clips_dir / "metadata" / "sprsound_clips_dataset.csv"

ast_clip_metadata = pd.read_csv(ast_metadata_csv)
ast_clip_metadata["clip_key"] = ast_clip_metadata["file_path"].str.replace(  # ty: ignore
    ".pt", "", regex=False
)

train_clip_rows, eval_clip_rows = train_test_split(
    ast_clip_metadata,
    test_size=0.2,
    random_state=42,
    stratify=ast_clip_metadata["target_label"],
)
ast_train_metadata = cap_majority_ratio(train_clip_rows)
ast_eval_metadata = cap_majority_ratio(eval_clip_rows)
train_clip_keys = set(ast_train_metadata["clip_key"])
eval_clip_keys = set(ast_eval_metadata["clip_key"])

ast_train_metadata_csv = metadata_dir / "ast_train_metadata.csv"
ast_eval_metadata_csv = metadata_dir / "ast_eval_metadata.csv"
ast_train_metadata.drop(columns=["clip_key"]).to_csv(
    ast_train_metadata_csv, index=False
)
ast_eval_metadata.drop(columns=["clip_key"]).to_csv(ast_eval_metadata_csv, index=False)

(
    ast_train_metadata["target_label"].value_counts(),
    ast_eval_metadata["target_label"].value_counts(),
)

(target_label
 pathological    276
 healthy         138
 Name: count, dtype: int64,
 target_label
 pathological    68
 healthy         34
 Name: count, dtype: int64)

Train AST using the provided hyperparameter file.

In [9]:
ast_run_name = f"ast_sprsound_{run_id}"
ast_model_path = model_dir / "ast" / ast_run_name / "final_model"

!python {scripts_dir}/train_ast.py \
    --train_features_dir {ast_features_dir} \
    --train_metadata {ast_train_metadata_csv} \
    --eval_features_dir {ast_features_dir} \
    --eval_metadata {ast_eval_metadata_csv} \
    --config_path {ast_hparams_path} \
    --output_dir {model_dir / "ast"} \
    --run_name {ast_run_name}

Using device: cuda

[DEBUG] Loading training configuration...
[DEBUG] Loaded configuration from /workspace/transformers-paper/hparams/ast_asthma.yaml

[DEBUG] Using device: cuda
[DEBUG] CUDA device count: 1
[DEBUG] Current CUDA device: 0
[DEBUG] Device name: NVIDIA A40

[DEBUG] Run name: ast_sprsound_20260603_165557
[DEBUG] Output directory: /workspace/transformers-paper/temp/sprsound_training/20260603_165557/models/ast/ast_sprsound_20260603_165557

[DEBUG] Dataset configuration:
[DEBUG] Training dataset size: 414
[DEBUG] Evaluation dataset size: 102
[DEBUG] Audio clip length: 5 seconds

[DEBUG] Initializing model...
config.json: 26.8kB [00:00, 43.6MB/s]
model.safetensors: 100%|██████████████████████| 346M/346M [00:01<00:00, 176MB/s]
Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the ch

In [10]:
assert ast_model_path.exists(), ast_model_path
ast_model_path

PosixPath('/workspace/transformers-paper/temp/sprsound_training/20260603_165557/models/ast/ast_sprsound_20260603_165557/final_model')

Run AST inference on the held-out SPRSound evaluation clips and evaluate the clip-level predictions.

In [11]:
ast_predictions_csv = predictions_dir / "ast_predictions.csv"
ast_metrics_json = predictions_dir / "ast_metrics.json"

!python {scripts_dir}/ast_inference.py \
    --model_path {ast_model_path} \
    --features_dir {ast_features_dir} \
    --metadata_csv {ast_eval_metadata_csv} \
    --output_mode csv \
    > {ast_predictions_csv}

!python {scripts_dir}/evaluate_ast.py \
    --predictions_csv {ast_predictions_csv} \
    --metadata_csv {ast_eval_metadata_csv} \
    --positive_label pathological \
    --negative_label healthy \
    > {ast_metrics_json}

pd.read_json(ast_metrics_json, typ="series")  # ty: ignore

Using device: cuda
Running inference: 100%|██████████████████████████| 4/4 [00:02<00:00,  1.60it/s]


accuracy        0.725490
sensitivity     0.823529
specificity     0.529412
precision       0.777778
f1              0.800000
youden          0.352941
tp             56.000000
fp             16.000000
tn             18.000000
fn             12.000000
dtype: float64